# Exercise 5: Xây dựng Data Pipeline thực tế với Apache Spark

Bài tập này hướng dẫn bạn xây dựng một pipeline xử lý dữ liệu hoàn chỉnh bao gồm:
- Đánh giá chất lượng dữ liệu
- Làm sạch và chuẩn hóa dữ liệu
- Phân tích nâng cao (RFM, Product Metrics)
- Tự động hóa pipeline và lưu kết quả
- Giám sát hiệu năng pipeline

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, when, isnull, lit, round, to_date, year, month,
    quarter, dayofweek, datediff, current_date, max as spark_max,
    min as spark_min, sum as spark_sum, avg as spark_avg,
    stddev as spark_stddev, countDistinct, concat_ws, coalesce
)
from pyspark.sql.types import StringType, IntegerType, DoubleType, DateType

spark = SparkSession.builder \
    .appName("Exercise5") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

print("SparkSession đã được tạo thành công.")
print(f"Spark version: {spark.version}")

## Giải thích Cell 1: Khởi tạo SparkSession và Import thư viện

**Các hàm/thư viện sử dụng:**
- `SparkSession.builder`: Builder pattern để cấu hình Spark session
- `.appName("Exercise5")`: Đặt tên ứng dụng hiển thị trên Spark UI
- `.master("spark://spark-master:7077")`: Kết nối tới Spark standalone cluster
- `.getOrCreate()`: Tạo session mới hoặc tái sử dụng session hiện có
- Các hàm từ `pyspark.sql.functions`: được import sẵn để dùng ở các cell sau

**Output mong đợi:**
- In ra thông báo "SparkSession đã được tạo thành công."
- In ra phiên bản Spark đang sử dụng
- Không có lỗi nào xuất hiện

In [ ]:
def assess_data_quality(df, table_name):
    """Đánh giá chất lượng dữ liệu của một DataFrame."""
    # Đếm tổng số dòng
    total_rows = df.count()
    
    # Đếm tổng số cột
    total_cols = len(df.columns)
    
    # Kiểm tra nulls per column
    null_counts = df.select(
        [count(when(col(c).isNull(), c)).alias(c) for c in df.columns]
    ).collect()[0].asDict()
    
    # Tính số dòng trùng lặp
    distinct_rows = df.distinct().count()
    duplicates = total_rows - distinct_rows
    
    # In schema
    print(f"\n=== Data Quality Report: {table_name} ===")
    print(f"Total rows: {total_rows}")
    print(f"Total columns: {total_cols}")
    print(f"Distinct rows: {distinct_rows}")
    print(f"Duplicates: {duplicates}")
    print(f"\nNull counts per column:")
    for col_name, null_count in null_counts.items():
        print(f"  {col_name}: {null_count}")
    print(f"\nSchema:")
    df.printSchema()
    
    return {
        "total_rows": total_rows,
        "null_summary": null_counts,
        "duplicates": duplicates,
        "distinct_rows": distinct_rows
    }

## Giải thích Cell 3: Hàm assess_data_quality

**Cú pháp và hàm sử dụng:**
- `df.count()`: Đếm tổng số dòng trong DataFrame (trigger action)
- `len(df.columns)`: Lấy số lượng cột từ danh sách tên cột
- `count(when(col(c).isNull(), c))`: Đếm số giá trị null của cột c
  - `col(c).isNull()`: Kiểm tra giá trị có phải null không
  - `when(condition, value)`: Trả về value nếu condition đúng, ngược lại null
  - `count(...)`: Chỉ đếm các giá trị khác null, nên chỉ đếm được khi isNull() = true
- `.collect()[0].asDict()`: Thu thập kết quả về driver và chuyển thành dict
- `df.distinct().count()`: Đếm số dòng duy nhất (loại bỏ trùng lặp)
- `df.printSchema()`: In cấu trúc schema của DataFrame

**Output mong đợi:**
- Hàm trả về dictionary chứa total_rows, null_summary, duplicates, distinct_rows
- In ra báo cáo chất lượng dữ liệu chi tiết cho bảng được truyền vào
- Chưa có output thực tế vì chưa gọi hàm

In [ ]:
# Load sales_data.csv
sales_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/home/jovyan/data/csv/sales_data.csv")

# Load customers.csv
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/home/jovyan/data/csv/customers.csv")

# Load products.csv
products_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/home/jovyan/data/csv/products.csv")

print("Đã load xong 3 file CSV.")
print(f"Sales: {sales_df.count()} dòng")
print(f"Customers: {customers_df.count()} dòng")
print(f"Products: {products_df.count()} dòng")

## Giải thích Cell 5: Load dữ liệu từ CSV

**Cú pháp sử dụng:**
- `spark.read.csv(path)`: Đọc file CSV thành DataFrame
- `.option("header", "true")`: Dòng đầu tiên là header (tên cột)
- `.option("inferSchema", "true")`: Tự động suy luận kiểu dữ liệu cho từng cột

**Output mong đợi:**
- Ba DataFrame: sales_df, customers_df, products_df được tạo thành công
- In ra số dòng của mỗi bảng
- Nếu file không tồn tại hoặc sai đường dẫn sẽ báo lỗi FileNotFoundException

In [ ]:
# Đánh giá chất lượng dữ liệu cho từng bảng
sales_quality = assess_data_quality(sales_df, "sales_data")
customers_quality = assess_data_quality(customers_df, "customers")
products_quality = assess_data_quality(products_df, "products")

## Giải thích Cell 7: Gọi hàm assess_data_quality

**Cú pháp sử dụng:**
- Gọi hàm `assess_data_quality(df, table_name)` cho từng DataFrame
- Kết quả trả về được lưu vào biến tương ứng: sales_quality, customers_quality, products_quality

**Output mong đợi:**
- In ra 3 báo cáo chất lượng dữ liệu liên tiếp
- Mỗi báo cáo gồm: total rows, columns, distinct rows, duplicates, null counts, schema
- Giúp xác định vấn đề chất lượng dữ liệu trước khi làm sạch

In [ ]:
def clean_sales_data(sales_df):
    """Làm sạch dữ liệu bán hàng."""
    # Lọc bỏ bản ghi không hợp lệ
    cleaned = sales_df.filter(
        (col("total_amount") > 0) &
        (col("quantity") > 0) &
        (col("unit_price") > 0)
    )
    
    # Chuẩn hóa tên region
    cleaned = cleaned.withColumn(
        "region",
        when(col("region").isin(["North", "NORTH", "north"]), "North")
        .when(col("region").isin(["South", "SOUTH", "south"]), "South")
        .when(col("region").isin(["East", "EAST", "east"]), "East")
        .when(col("region").isin(["West", "WEST", "west"]), "West")
        .otherwise(col("region"))
    )
    
    # Thêm các cột derived
    cleaned = cleaned \
        .withColumn("date_formatted", to_date(col("date"))) \
        .withColumn("year", year(col("date_formatted"))) \
        .withColumn("month", month(col("date_formatted"))) \
        .withColumn("quarter", quarter(col("date_formatted"))) \
        .withColumn("day_of_week", dayofweek(col("date_formatted"))) \
        .withColumn("is_weekend", when(col("day_of_week").isin([1, 7]), True).otherwise(False)) \
        .withColumn("revenue_per_unit", round(col("total_amount") / col("quantity"), 2))
    
    # Validate calculated_total vs total_amount
    cleaned = cleaned.withColumn(
        "calculated_total",
        round(col("quantity") * col("unit_price"), 2)
    )
    
    # Drop calculated_total sau khi validate
    cleaned = cleaned.drop("calculated_total")
    
    return cleaned

## Giải thích Cell 9: Hàm clean_sales_data

**Cú pháp và hàm sử dụng:**
- `df.filter(condition)`: Lọc dòng thỏa mãn điều kiện
  - Toán tử `&` thay vì `and` trong PySpark filter
- `df.withColumn(name, expr)`: Thêm hoặc thay thế cột
- `when(condition, value).otherwise(default)`: Tương đương CASE WHEN trong SQL
- `col("region").isin([...])`: Kiểm tra giá trị có nằm trong danh sách không
- `to_date(col)`: Chuyển string sang date type
- `year()`, `month()`, `quarter()`, `dayofweek()`: Trích xuất thành phần từ date
- `round(expr, n)`: Làm tròn đến n chữ số thập phân
- `df.drop(col_name)`: Xóa cột khỏi DataFrame

**Output mong đợi:**
- Trả về DataFrame đã được làm sạch:
  - Loại bỏ dòng có total_amount/quantity/unit_price <= 0
  - Region được chuẩn hóa về dạng title case
  - Có thêm các cột: date_formatted, year, month, quarter, day_of_week, is_weekend, revenue_per_unit
- Chưa có output vì chưa gọi hàm

In [ ]:
def enrich_customer_data(customers_df):
    """Làm giàu dữ liệu khách hàng."""
    enriched = customers_df \
        .withColumn("signup_date_formatted", to_date(col("signup_date"))) \
        .withColumn("account_age_days", datediff(current_date(), col("signup_date_formatted"))) \
        .withColumn("account_age_years", round(col("account_age_days") / 365.25, 1))
    
    # Phân loại generation dựa trên năm sinh
    enriched = enriched.withColumn(
        "generation",
        when(col("birth_year") >= 1997, "Gen Z")
        .when(col("birth_year") >= 1981, "Millennial")
        .when(col("birth_year") >= 1965, "Gen X")
        .otherwise("Boomer")
    )
    
    # Chuyển loyalty_tier sang numeric
    enriched = enriched.withColumn(
        "loyalty_tier_numeric",
        when(col("loyalty_tier") == "Bronze", 1)
        .when(col("loyalty_tier") == "Silver", 2)
        .when(col("loyalty_tier") == "Gold", 3)
        .when(col("loyalty_tier") == "Platinum", 4)
        .otherwise(0)
    )
    
    return enriched

## Giải thích Cell 11: Hàm enrich_customer_data

**Cú pháp và hàm sử dụng:**
- `datediff(date1, date2)`: Tính số ngày chênh lệch giữa hai ngày
- `current_date()`: Trả về ngày hiện tại (hàm Spark, không phải Python)
- `round(expr, 1)`: Làm tròn 1 chữ số thập phân cho account_age_years
- Chuỗi `when().when().otherwise()`: Phân loại generation theo birth_year
  - Gen Z: >= 1997
  - Millennial: >= 1981
  - Gen X: >= 1965
  - Boomer: < 1965
- Ánh xạ loyalty_tier sang số: Bronze=1, Silver=2, Gold=3, Platinum=4

**Output mong đợi:**
- Trả về DataFrame customers đã được làm giàu:
  - signup_date_formatted: dạng date
  - account_age_days: số ngày kể từ đăng ký
  - account_age_years: số năm (làm tròn 1 chữ số)
  - generation: nhãn thế hệ
  - loyalty_tier_numeric: bậc trung thành dạng số

In [ ]:
# Áp dụng làm sạch và so sánh số dòng
clean_sales = clean_sales_data(sales_df)
enriched_customers = enrich_customer_data(customers_df)

print(f"Sales - Original: {sales_df.count()}, Cleaned: {clean_sales.count()}")
print(f"Customers - Original: {customers_df.count()}, Enriched: {enriched_customers.count()}")

## Giải thích Cell 13: Áp dụng hàm làm sạch

**Cú pháp sử dụng:**
- Gọi `clean_sales_data(sales_df)` và gán vào `clean_sales`
- Gọi `enrich_customer_data(customers_df)` và gán vào `enriched_customers`
- So sánh count trước và sau khi làm sạch

**Output mong đợi:**
- In ra số dòng original vs cleaned của sales
  - Số dòng cleaned có thể ít hơn do lọc bỏ bản ghi không hợp lệ
- In ra số dòng original vs enriched của customers
  - Số dòng giữ nguyên vì enrich chỉ thêm cột, không lọc

In [ ]:
def calculate_customer_metrics(sales_df, customers_df):
    """Tính toán metrics và phân khúc khách hàng RFM."""
    # Aggregate theo customer_id
    customer_agg = sales_df.groupBy("customer_id").agg(
        count("*").alias("transaction_count"),
        spark_sum("total_amount").alias("total_spent"),
        spark_avg("total_amount").alias("avg_transaction_value"),
        spark_min("date_formatted").alias("first_purchase_date"),
        spark_max("date_formatted").alias("last_purchase_date"),
        countDistinct("category").alias("unique_categories"),
        countDistinct("product_name").alias("unique_products"),
        spark_stddev("total_amount").alias("spending_stddev")
    )
    
    # Lấy current_date từ max(date_formatted)
    max_date_row = sales_df.select(spark_max("date_formatted")).collect()[0][0]
    
    # Tính RFM metrics
    customer_metrics = customer_agg \
        .withColumn("recency_days", datediff(lit(max_date_row), col("last_purchase_date"))) \
        .withColumn("frequency", col("transaction_count")) \
        .withColumn("monetary", col("total_spent")) \
        .withColumn(
            "avg_days_between_purchases",
            when(col("transaction_count") > 1,
                 round(datediff(col("last_purchase_date"), col("first_purchase_date")) / (col("transaction_count") - 1), 1))
            .otherwise(None)
        )
    
    # Join với customers_df
    customer_metrics = customer_metrics.join(customers_df, "customer_id", "left")
    
    # Tính rfm_score (đơn giản hóa: chia percentile thành điểm 1-5)
    # Ở đây dùng cách tính đơn giản cho intern
    customer_metrics = customer_metrics \
        .withColumn("r_score", when(col("recency_days") <= 30, 5)
                    .when(col("recency_days") <= 90, 4)
                    .when(col("recency_days") <= 180, 3)
                    .when(col("recency_days") <= 365, 2)
                    .otherwise(1)) \
        .withColumn("f_score", when(col("frequency") >= 10, 5)
                    .when(col("frequency") >= 7, 4)
                    .when(col("frequency") >= 4, 3)
                    .when(col("frequency") >= 2, 2)
                    .otherwise(1)) \
        .withColumn("m_score", when(col("monetary") >= 5000, 5)
                    .when(col("monetary") >= 3000, 4)
                    .when(col("monetary") >= 1000, 3)
                    .when(col("monetary") >= 500, 2)
                    .otherwise(1)) \
        .withColumn("rfm_score", col("r_score") + col("f_score") + col("m_score"))
    
    # Phân khúc theo rfm_score
    customer_segments = customer_metrics.withColumn(
        "segment",
        when(col("rfm_score") >= 13, "Champions")
        .when(col("rfm_score") >= 11, "Loyal Customers")
        .when(col("rfm_score") >= 9, "Potential Loyalists")
        .when(col("rfm_score") >= 7, "At Risk")
        .otherwise("Lost")
    )
    
    return customer_segments

## Giải thích Cell 15: Hàm calculate_customer_metrics (RFM)

**Cú pháp và hàm sử dụng:**
- `groupBy("customer_id").agg(...)`: Nhóm theo customer_id và tính aggregate
- `count("*")`: Đếm số giao dịch
- `spark_sum`, `spark_avg`, `spark_min`, `spark_max`: Các hàm aggregate
- `countDistinct(col)`: Đếm số giá trị duy nhất
- `spark_stddev(col)`: Độ lệch chuẩn
- `lit(value)`: Tạo cột hằng số (dùng max_date_row làm mốc)
- `datediff(date1, date2)`: Tính recency_days
- `.join(df, on_col, how)`: Left join với customers_df
- Chuỗi when để tính R/F/M score (1-5) và rfm_score = R+F+M
- Phân khúc: Champions >=13, Loyal >=11, Potential >=9, At Risk >=7, Lost <7

**Output mong đợi:**
- Trả về DataFrame customer_segments với đầy đủ metrics và segment label
- Chưa có output vì chưa gọi hàm

In [ ]:
def calculate_product_metrics(sales_df, products_df):
    """Tính toán metrics cho sản phẩm."""
    # Aggregate theo category + product_name
    product_agg = sales_df.groupBy("category", "product_name").agg(
        count("*").alias("transaction_count"),
        spark_sum("quantity").alias("total_quantity_sold"),
        spark_sum("total_amount").alias("total_revenue"),
        spark_avg("total_amount").alias("avg_transaction_value"),
        countDistinct("customer_id").alias("unique_customers"),
        countDistinct("region").alias("unique_regions")
    )
    
    # Join với products_df để lấy cost
    product_metrics = product_agg.join(
        products_df.select("product_name", "cost", "price"),
        "product_name",
        "left"
    )
    
    # Tính profit metrics
    product_metrics = product_metrics \
        .withColumn("total_cost", round(col("cost") * col("total_quantity_sold"), 2)) \
        .withColumn("total_profit", round(col("total_revenue") - col("total_cost"), 2)) \
        .withColumn(
            "profit_margin_actual",
            round((col("total_profit") / col("total_revenue")) * 100, 2)
        ) \
        .withColumn(
            "roi_percent",
            round((col("total_profit") / col("total_cost")) * 100, 2)
        )
    
    return product_metrics

## Giải thích Cell 17: Hàm calculate_product_metrics

**Cú pháp và hàm sử dụng:**
- `groupBy("category", "product_name").agg(...)`: Nhóm theo 2 cột
- `spark_sum("quantity")`: Tổng số lượng bán
- `spark_sum("total_amount")`: Tổng doanh thu
- `countDistinct("customer_id")`: Số khách hàng duy nhất mua sản phẩm
- `countDistinct("region")`: Số khu vực có bán sản phẩm
- `.join(products_df.select(...), "product_name", "left")`: Join để lấy cost và price
- `total_cost = cost * total_quantity_sold`
- `total_profit = total_revenue - total_cost`
- `profit_margin_actual = (total_profit / total_revenue) * 100`
- `roi_percent = (total_profit / total_cost) * 100`

**Output mong đợi:**
- Trả về DataFrame product_metrics với các chỉ số lợi nhuận
- Chưa có output vì chưa gọi hàm

In [ ]:
# Gọi cả hai hàm metrics
customer_segments = calculate_customer_metrics(clean_sales, enriched_customers)
product_metrics = calculate_product_metrics(clean_sales, products_df)

# Hiển thị tóm tắt phân khúc khách hàng
print("=== Customer Segmentation Summary ===")
customer_segments.groupBy("segment").count().orderBy(col("count").desc()).show(truncate=False)

# Hiển thị top 10 sản phẩm theo doanh thu
print("=== Top 10 Products by Revenue ===")
product_metrics.orderBy(col("total_revenue").desc()).show(10, truncate=False)

## Giải thích Cell 19: Gọi hàm và hiển thị kết quả

**Cú pháp sử dụng:**
- Gọi `calculate_customer_metrics(clean_sales, enriched_customers)`
- Gọi `calculate_product_metrics(clean_sales, products_df)`
- `groupBy("segment").count()`: Đếm số khách hàng mỗi phân khúc
- `.orderBy(col("count").desc())`: Sắp xếp giảm dần theo count
- `.show(truncate=False)`: Hiển thị bảng không cắt ngắn nội dung
- `product_metrics.orderBy(...).show(10)`: Hiển thị top 10 sản phẩm

**Output mong đợi:**
- Bảng phân khúc khách hàng: Champions, Loyal, Potential, At Risk, Lost kèm số lượng
- Bảng top 10 sản phẩm với đầy đủ metrics: revenue, profit, margin, roi

In [ ]:
def create_summary_tables(clean_sales, customers_df, products_df):
    """Tạo các bảng tóm tắt cho pipeline."""
    from pyspark.sql.functions import current_timestamp
    
    # Daily summary: groupBy date_formatted + region
    daily_summary = clean_sales.groupBy("date_formatted", "region").agg(
        count("*").alias("transaction_count"),
        spark_sum("total_amount").alias("daily_revenue"),
        spark_avg("total_amount").alias("avg_transaction_value")
    ).withColumn("created_timestamp", current_timestamp())
    
    # Monthly category: groupBy year + month + category
    monthly_category = clean_sales.groupBy("year", "month", "category").agg(
        count("*").alias("transaction_count"),
        spark_sum("total_amount").alias("monthly_revenue"),
        spark_sum("quantity").alias("total_quantity")
    ).withColumn("created_timestamp", current_timestamp())
    
    # Loyalty trends: join customers, groupBy year + month + loyalty_tier
    sales_with_loyalty = clean_sales.join(
        customers_df.select("customer_id", "loyalty_tier"),
        "customer_id",
        "left"
    )
    loyalty_trends = sales_with_loyalty.groupBy("year", "month", "loyalty_tier").agg(
        count("*").alias("transaction_count"),
        spark_sum("total_amount").alias("monthly_revenue")
    ).withColumn("created_timestamp", current_timestamp())
    
    return daily_summary, monthly_category, loyalty_trends

## Giải thích Cell 21: Hàm create_summary_tables

**Cú pháp và hàm sử dụng:**
- `current_timestamp()`: Trả về timestamp hiện tại (dùng cho audit trail)
- `groupBy("date_formatted", "region").agg(...)`: Tóm tắt theo ngày và khu vực
- `groupBy("year", "month", "category").agg(...)`: Tóm tắt theo tháng và danh mục
- `.join(customers_df.select(...), "customer_id", "left")`: Join để lấy loyalty_tier
- `groupBy("year", "month", "loyalty_tier").agg(...)`: Xu hướng theo bậc trung thành
- `.withColumn("created_timestamp", ...)`: Thêm cột timestamp vào mỗi bảng

**Output mong đợi:**
- Trả về tuple 3 DataFrame: daily_summary, monthly_category, loyalty_trends
- Mỗi bảng có cột created_timestamp để biết thời gian tạo
- Chưa có output vì chưa gọi hàm

In [ ]:
def save_pipeline_results(customer_segments, product_metrics, daily_summary, monthly_category, loyalty_trends):
    """Lưu kết quả pipeline ra parquet và CSV."""
    base_path = "/home/jovyan/data/parquet/"
    
    # Lưu tất cả dưới dạng parquet
    customer_segments.write.mode("overwrite").parquet(base_path + "customer_segments")
    product_metrics.write.mode("overwrite").parquet(base_path + "product_metrics")
    daily_summary.write.mode("overwrite").parquet(base_path + "daily_summary")
    monthly_category.write.mode("overwrite").parquet(base_path + "monthly_category")
    loyalty_trends.write.mode("overwrite").parquet(base_path + "loyalty_trends")
    
    # Lưu top customers report và product performance report dưới dạng CSV
    customer_segments.coalesce(1).write.mode("overwrite") \
        .option("header", "true").csv(base_path + "top_customers_report")
    
    product_metrics.coalesce(1).write.mode("overwrite") \
        .option("header", "true").csv(base_path + "product_performance_report")
    
    print("Đã lưu xong tất cả kết quả pipeline.")

## Giải thích Cell 23: Hàm save_pipeline_results

**Cú pháp và hàm sử dụng:**
- `df.write.mode("overwrite").parquet(path)`: Ghi DataFrame ra định dạng Parquet
  - `mode("overwrite")`: Ghi đè nếu thư mục đã tồn tại
- `df.coalesce(1)`: Gộp tất cả partition thành 1 file duy nhất
  - Dùng cho CSV report để dễ đọc bằng Excel
- `.option("header", "true").csv(path)`: Ghi CSV có header

**Output mong đợi:**
- Tạo 5 thư mục parquet trong /home/jovyan/data/parquet/
- Tạo 2 thư mục CSV (top_customers_report, product_performance_report)
- In thông báo hoàn thành
- Chưa có output vì chưa gọi hàm

In [ ]:
# Chạy toàn bộ pipeline
daily_summary, monthly_category, loyalty_trends = create_summary_tables(
    clean_sales, enriched_customers, products_df
)

save_pipeline_results(
    customer_segments, product_metrics,
    daily_summary, monthly_category, loyalty_trends
)

print("Pipeline Task 5.4 hoàn thành.")

## Giải thích Cell 25: Chạy pipeline

**Cú pháp sử dụng:**
- Gọi `create_summary_tables()` với 3 DataFrame đầu vào
- Unpack tuple trả về thành 3 biến riêng biệt
- Gọi `save_pipeline_results()` với 5 DataFrame

**Output mong đợi:**
- In "Đã lưu xong tất cả kết quả pipeline."
- In "Pipeline Task 5.4 hoàn thành."
- Các file parquet và CSV được tạo trên disk

In [ ]:
def monitor_pipeline_performance():
    """Giám sát hiệu năng pipeline."""
    # Kiểm tra partition sizes
    partition_sizes = clean_sales.rdd.glom().map(len).collect()
    
    min_size = min(partition_sizes)
    max_size = max(partition_sizes)
    num_partitions = len(partition_sizes)
    skew_ratio = max_size / min_size if min_size > 0 else float('inf')
    
    print("=== Pipeline Performance Monitor ===")
    print(f"Number of partitions: {num_partitions}")
    print(f"Partition sizes: {partition_sizes}")
    print(f"Min partition size: {min_size}")
    print(f"Max partition size: {max_size}")
    print(f"Skew ratio (max/min): {skew_ratio:.2f}")
    
    # Explain last query plan
    print("\n=== Last Query Plan ===")
    clean_sales.explain()
    
    # Optimization recommendations
    print("\n=== Optimization Recommendations ===")
    if skew_ratio > 3:
        print("WARNING: High data skew detected (ratio > 3).")
        print("  -> Consider repartitioning by a more balanced key.")
        print("  -> Use salting technique for skewed joins.")
    else:
        print("Data distribution looks balanced.")
    
    # So sánh số core vs số partition
    spark_conf = spark.sparkContext.getConf()
    cores = int(spark_conf.get("spark.executor.cores", "1"))
    executors = int(spark_conf.get("spark.executor.instances", "1"))
    total_cores = cores * executors
    
    print(f"\nCluster info: {executors} executors x {cores} cores = {total_cores} total cores")
    print(f"Partitions: {num_partitions}")
    
    if num_partitions < total_cores:
        print(f"  -> Under-utilization: {num_partitions} partitions < {total_cores} cores.")
        print(f"     Recommend repartition({total_cores}) or higher.")
    elif num_partitions > total_cores * 3:
        print(f"  -> Over-partitioned: {num_partitions} partitions >> {total_cores} cores.")
        print(f"     Consider coalesce to reduce overhead.")
    else:
        print("  -> Partition count is reasonable for cluster size.")

## Giải thích Cell 27: Hàm monitor_pipeline_performance

**Cú pháp và hàm sử dụng:**
- `df.rdd.glom()`: Chuyển mỗi partition thành một list (array)
- `.map(len)`: Tính kích thước (số phần tử) của mỗi partition
- `.collect()`: Thu thập kết quả về driver
- `skew_ratio = max / min`: Tỷ lệ chênh lệch, >3 là đáng báo động
- `df.explain()`: In execution plan của DataFrame (physical plan)
- `spark.sparkContext.getConf().get(key, default)`: Lấy cấu hình Spark
- So sánh num_partitions vs total_cores để đưa khuyến nghị

**Output mong đợi:**
- In thông tin partition: số lượng, kích thước min/max, skew ratio
- In execution plan của clean_sales
- In khuyến nghị tối ưu dựa trên skew ratio và cluster config
- Chưa có output vì chưa gọi hàm

In [ ]:
# Gọi hàm giám sát hiệu năng
monitor_pipeline_performance()

## Giải thích Cell 29: Gọi monitor_pipeline_performance

**Cú pháp sử dụng:**
- Gọi hàm `monitor_pipeline_performance()` không tham số
- Hàm sử dụng biến global `clean_sales` và `spark` từ các cell trước

**Output mong đợi:**
- Báo cáo hiệu năng pipeline gồm:
  - Số partition và kích thước từng partition
  - Skew ratio và cảnh báo nếu > 3
  - Execution plan của clean_sales
  - Thông tin cluster (executors, cores)
  - Khuyến nghị tối ưu cụ thể

---

**Hoàn thành Exercise 5!** Bạn đã xây dựng một data pipeline hoàn chỉnh với Spark bao gồm đánh giá chất lượng, làm sạch, phân tích RFM, tự động hóa và giám sát hiệu năng.